# Example 01.05 — create the batch-scoring pipeline

Builds a fixed scoring workflow from the exact model and fitted AutoFE state created
by notebook 01.04. It never refits transformations on scoring data.

This JSON is a reproducible automation example. Use the frontend's pipeline editor
to design or infer more flexible production workflows.


In [1]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "ml_app_client").is_dir()),
    None,
)
if REPOSITORY_ROOT is None:
    raise RuntimeError("Start Jupyter inside the ml-app repository")
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from ml_app_client import MLAppClient, ResourceNotFoundError

client = MLAppClient.connect()
print("Connected to ML App")


Connected to ML App


## Choose resource names

These are normal names passed to `ml_app_client`. Edit them directly and use the
same values in the following notebooks. Dataset and pipeline names are resolved
inside the selected Business Case. Business Case and service names are globally
unique on one ML App installation.


In [2]:
# These are ordinary platform resource names. Change the two globally unique
# names (Business Case and model service) when sharing one installation.
BUSINESS_CASE_NAME = "[MLAPP EXAMPLE 01 v2] Estates Lifecycle - demo"
TRAINING_DATASET_NAME = "Example01 Estates - Training"
SCORING_DATASET_NAME = "Example01 Estates - Batch Input"
ACTUALS_DATASET_NAME = "Example01 Estates - Actuals"
TRAINING_PIPELINE_NAME = "Example01 03 - AutoML Training"
BATCH_PIPELINE_NAME = "Example01 05 - Batch Scoring"
MONITORING_PIPELINE_NAME = "Example01 07 - Performance Monitoring"
MODEL_NAME = "Example01 Estates Price Model"
OUTPUT_NAME_PREFIX = "Example01 Estates AutoML"
MODEL_SERVICE_NAME = "Example01 10 - Estates Model Service - demo"

TRAINING_RUN_KEY = "Example01-training-v2"
BATCH_RUN_KEY = "Example01-batch-scoring-v2"
MONITORING_RUN_KEY = "Example01-monitoring-v2"

print("Resource names configured for:", BUSINESS_CASE_NAME)


Resource names configured for: [MLAPP EXAMPLE 01 v2] Estates Lifecycle - demo


In [3]:
from examples.example01_lifecycle import build_batch_scoring_definition

model = client.model_by_name(
    business_case_name=BUSINESS_CASE_NAME,
    model_name=MODEL_NAME,
)
scoring = client.dataset_by_name(
    business_case_name=BUSINESS_CASE_NAME,
    dataset_name=SCORING_DATASET_NAME,
)
definition = build_batch_scoring_definition(
    model,
    scoring.logical_id,
    output_name_prefix=OUTPUT_NAME_PREFIX,
)
try:
    pipeline = client.pipeline_by_name(business_case_name=BUSINESS_CASE_NAME, pipeline_name=BATCH_PIPELINE_NAME)
    version = client.latest_published_pipeline_version(str(pipeline["id"]))
    created = False
except ResourceNotFoundError:
    business_case = client.business_case_by_name(BUSINESS_CASE_NAME)
    pipeline = client.create_pipeline(business_case_id=str(business_case["id"]), name=BATCH_PIPELINE_NAME, description="Example01 full-scope batch inference with a pinned training bundle.", pipeline_type="batch_scoring", definition=definition)
    version = client.publish_pipeline_draft(str(pipeline["id"]))
    created = True
print(
    f"{'CREATED' if created else 'FOUND'} {pipeline['name']}; "
    f"published version={version['version_number']}; model={model['id']}"
)


CREATED Example01 05 - Batch Scoring; published version=1; model=28c00b2c-4ded-4715-8a19-972168070e7f
